# Healthy canonical spatial atlas first; probability flow as an ablation

**Hypothesis A:** one healthy PT realization teaches a molecular atlas that locates profiles from another healthy realization using expression alone. **Hypothesis B:** distributional evolution adds predictive value beyond movement of that atlas's mean.

The previous OT-distilled affine flow lost to training-centroid displacement. That result does not gate the atlas hypothesis. We now fit $X_{si}=\mu(z_{si})+b_s+\epsilon_{si}$ with a training-only kernel mean curve and a restricted global-intercept sensitivity. The primary coordinate is $\arg\min_z\|Y_i-\mu(z)\|^2$. The other control supplies neither DPT nor S1/S2/S3 labels to projection. Train on Ctrl1A2 → project Ctrl1A4, then swap, rotating held-out gene folds.

A separate probabilistic atlas models $\rho_z=\mathcal N(\mu(z),\Sigma(z))$. Its Gaussian velocity is $v(y,z)=\mu'(z)+A(z)(y-\mu(z))$, with the symmetric solution of $A\Sigma+\Sigma A=\Sigma'$. Test covariance evolution against the **same smooth mean shift**. Gaussian likelihood mapping and Gaussian transport are separate experiments: an exact marginal-preserving field does not, by itself, add another position likelihood to the same Gaussian atlas.

Two controls support descriptive comparisons only. The training coordinate remains the existing marker-informed DPT scaffold; this is an inductive atlas test, not a wholly DPT-free training method or a unique spatial metric. Test DPT is a post-projection comparator. Upstream labels are inherited annotations, not independent fine spatial truth. Total-count normalization retains compositional dependence on withheld genes. x/y, glomerular depth and withheld markers never enter reconstruction. AKI and TF interpretation stay deferred until healthy mapping is validated.

In [ ]:
from pathlib import Path
import argparse, hashlib, json, os, sys
import numpy as np
import pandas as pd
import anndata as ad
from scipy import sparse
from scipy.spatial import cKDTree
from scipy.stats import spearmanr, rankdata
from scipy.special import logsumexp
import matplotlib.pyplot as plt
from IPython.display import display
from IPython import get_ipython
_ipython = get_ipython()
if _ipython is not None:
    _ipython.run_line_magic('matplotlib', 'inline')
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(REPO))
from pseudospace.repeated_inputs import load_repeated_mouse_inputs, _read_and_validate_segmentation
from pseudospace.spatial_probability_flow import scaffold_centroid_rows, representation, window_clouds, benchmark
from pseudospace.canonical_mean_atlas import fit_mean_atlas, project_mean_atlas, fit_global_offset, conditional_gene_predictions, segment_means
from pseudospace.gaussian_spatial_flow import fit_gaussian_path, project_gaussian_path, gaussian_transport_benchmark
from scipy.interpolate import interp1d
from pseudospace.stage_cache import cached_frame, cached_payload
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', default=os.environ.get('PSEUDOSPACE_DATA_ROOT', str(REPO / 'data')))
parser.add_argument('--results-root', default=os.environ.get('PSEUDOSPACE_RESULTS_ROOT', str(REPO / 'results')))
args, _ = parser.parse_known_args()
DATA_ROOT, RESULTS_ROOT = Path(args.data_root).expanduser(), Path(args.results_root).expanduser()
DPT_PATH = Path(os.environ.get('PSEUDOSPACE_15_DPT_REFERENCE', str(RESULTS_ROOT / 'human_vs_healthy_mouse' / 'cross_species_pt_dpt.h5ad')))
DPT_COLUMN = os.environ.get('PSEUDOSPACE_15_DPT_COLUMN', 'total_scanpy_dpt')
GLOMERULUS_PATH = Path(os.environ.get('PSEUDOSPACE_15_GLOMERULI', str(RESULTS_ROOT / 'human_vs_healthy_mouse' / 'cross_species_harmony_pass1.h5ad')))
OUT = RESULTS_ROOT / 'spatial_probability_flow_proof_of_concept'
OUT.mkdir(parents=True, exist_ok=True)
CONTROLS = ('Ctrl1A2', 'Ctrl1A4')
NOTEBOOK_LOGIC_VERSION = '15.atlas_first.1'
SEED, N_WINDOWS, N_COMPONENTS = 15, 12, 15
MIN_COUNT, MAX_POINTS = 12, 128
RIDGE, EPSILON, ITERATIONS = 1., .5, 4
N_GENE_FOLDS = 3
N_BOOT = int(os.environ.get('PSEUDOSPACE_15_BOOTSTRAPS', '10'))
ATLAS_BANDWIDTH, OFFSET_PENALTY, COV_SHRINKAGE = .08, 4., .25
ATLAS_GRID = np.linspace(0., 1., 101)
if N_BOOT < 2:
    raise ValueError('At least two bootstrap repetitions are required')
MARKERS = set('Slc5a2 Slc5a12 Gatm Slc22a6 Slc13a3 Slc1a1 Cyp2e1 Slc7a13 Slc22a7 Slc5a1 Cyp7b1 Slc7a12 Lrp2 Cubn Slc34a1 Slc6a18 Acsm3 Agt'.casefold().split())
CODE_FILES = [REPO / 'pseudospace' / f for f in ('spatial_probability_flow.py', 'canonical_mean_atlas.py', 'gaussian_spatial_flow.py', 'repeated_inputs.py')]
CODE_DIGEST = hashlib.sha256(b''.join(p.read_bytes() for p in CODE_FILES)).hexdigest()
CACHE_PARAMS = dict(version=NOTEBOOK_LOGIC_VERSION, seed=SEED, windows=N_WINDOWS, components=N_COMPONENTS, min_count=MIN_COUNT, max_points=MAX_POINTS, ridge=RIDGE, epsilon=EPSILON, iterations=ITERATIONS, atlas_bandwidth=ATLAS_BANDWIDTH, offset_penalty=OFFSET_PENALTY, cov_shrinkage=COV_SHRINKAGE, atlas_grid=ATLAS_GRID.tolist(), gene_folds=N_GENE_FOLDS)
plt.rcParams.update({'font.size': 9, 'axes.spines.top': False, 'axes.spines.right': False, 'pdf.fonttype': 42, 'svg.fonttype': 'none'})

## 1. Validated healthy PT and training-coordinate availability

Load the two current control count matrices and kept-fine segmentations, verify IDs and centroids, apply the healthy-cohort pooled p5 spot-count structural floor, retain all common measured genes and normalize to 10,000 + log1p before PT selection. No tubule gene-count QC. Read the saved PT DPT by a strict unique per-specimen centroid match; feature IDs also agree when exported.

**Keep every validated healthy PT profile for held-out projection**, including those absent from DPT. Only DPT-available profiles in the training specimen fit the canonical curve. Fold training rescales that specimen's own supplied DPT min/max to [0,1]; no held-out coordinate determines its axis, grid, bandwidth or PCA. The joint healthy-scaled DPT below is used only for the historical transport diagnostic. Test-specimen DPT availability limits comparisons, not expression-only projection.

In [ ]:
healthy = load_repeated_mouse_inputs(DATA_ROOT, samples=CONTROLS)
obs = healthy.obs.copy()
genes = healthy.var_names.to_numpy(str)
ids = obs.structure_id.to_numpy(str)
specimens = obs['sample'].astype(str).to_numpy()
labels = obs.segment_class.astype(str).str.replace('PT-', '', regex=False).to_numpy()
if not np.isin(labels, ['S1', 'S2', 'S3']).all():
    raise ValueError('Resolve uncertain PT labels upstream; do not silently drop structures')
M = healthy.layers['lognorm']
X = M.toarray().astype(float) if sparse.issparse(M) else np.asarray(M, float)
if not np.isfinite(X).all():
    raise ValueError('Expression contains nonfinite values')
if not DPT_PATH.is_file():
    raise FileNotFoundError(f'Run the existing reviewed DPT workflow first, or supply PSEUDOSPACE_15_DPT_REFERENCE: {DPT_PATH}')
reference = ad.read_h5ad(DPT_PATH)
required = {'sample', 'x_centroid', 'y_centroid', DPT_COLUMN}
if not required.issubset(reference.obs):
    raise ValueError(f'DPT reference missing {required - set(reference.obs)}')
# Some existing exports omit feature_index. Use a strictly unique centroid match.
xy = obs[['x_centroid', 'y_centroid']].to_numpy(float)
row = scaffold_centroid_rows(xy, specimens, reference.obs[['x_centroid', 'y_centroid']].to_numpy(float), reference.obs['sample'].astype(str).to_numpy(), tolerance=1.)
if 'feature_index' in reference.obs:
    present = row >= 0
    ref_feature = pd.to_numeric(reference.obs.iloc[row[present]].feature_index, errors='raise').to_numpy(float)
    source_feature = obs.iloc[np.flatnonzero(present)].feature_index.to_numpy(float)
    if not np.array_equal(ref_feature, source_feature):
        raise ValueError('DPT feature indices disagree with validated centroid join')
# Record DPT availability without removing profiles from held-out projection.
eligible = row >= 0
excluded_scaffold = obs.loc[~eligible, ['structure_id', 'sample', 'segment_class']].copy()
excluded_scaffold['reason'] = 'absent from supplied PT DPT scaffold'
excluded_scaffold.to_csv(OUT / 'scaffold_ineligible_structures.csv', index=False)
display(pd.crosstab(specimens, eligible, colnames=['DPT eligible']))
z_raw = np.full(len(X), np.nan)
z_raw[eligible] = reference.obs[DPT_COLUMN].to_numpy(float)[row[eligible]]
if not np.isfinite(z_raw[eligible]).all() or np.ptp(z_raw[eligible]) <= 0:
    raise ValueError('Available DPT must be finite and nonconstant')
z_dpt = (z_raw-np.nanmin(z_raw)) / (np.nanmax(z_raw)-np.nanmin(z_raw))
marker_mask = np.array([g.casefold() in MARKERS for g in genes])
gene_fold = np.array([int(hashlib.sha256(g.casefold().encode()).hexdigest()[:8], 16) % N_GENE_FOLDS for g in genes])
model_genes = np.flatnonzero(~marker_mask & (gene_fold != 0))
eval_genes = np.flatnonzero(~marker_mask & (gene_fold == 0))
if min(len(model_genes), len(eval_genes)) < N_COMPONENTS:
    raise ValueError('Insufficient marker-excluded modeling/evaluation genes')
lo = max(np.nanmin(z_dpt[specimens == s]) for s in CONTROLS)
hi = min(np.nanmax(z_dpt[specimens == s]) for s in CONTROLS)
if hi <= lo:
    raise ValueError('No common DPT support')
centers = np.linspace(lo, hi, N_WINDOWS)
spacing = centers[1]-centers[0]
widths = {'overlapping': .8*spacing, 'nonoverlap': .499*spacing}
manifest = dict(healthy.uns['repeated_mouse_input_manifest'])
manifest.update(scaffold_eligible_structures=int(eligible.sum()), total_healthy_pt=len(X), scaffold_excluded_structures=len(excluded_scaffold), scaffold_join='unique within-specimen centroid within 1 pixel; feature_index checked when available', scaffold_path=str(DPT_PATH), scaffold_column=DPT_COLUMN, scaffold_sha256=hashlib.sha256(z_raw.tobytes()).hexdigest(), dpt_scaled_healthy_range=[float(np.nanmin(z_raw)), float(np.nanmax(z_raw))], common_support=[float(lo), float(hi)], markers_excluded=sorted(MARKERS), model_genes=genes[model_genes].tolist(), evaluation_genes=genes[eval_genes].tolist(), config=CACHE_PARAMS, code_digest=CODE_DIGEST)
(OUT / 'input_manifest.json').write_text(json.dumps(manifest, indent=2))
display(pd.crosstab(specimens, labels))
print('Dense matrix GiB:', X.nbytes/2**30, 'DPT scaffolding:', DPT_PATH.name)

## 2. Historical distribution path and OT-flow result

Retain the previous fixed-DPT diagnostic on the explicitly DPT-available cohort. Each representation is training-only; compare adjacent held-out distributions with identity, raw centroid increments, DPT-local kNN increments and OT-distilled flow. Overlapping and non-overlapping windows remain separate. Failure of this field records evidence about Hypothesis B and **never skips the atlas experiment**.

In [ ]:
train0 = (specimens == CONTROLS[0]) & eligible
test0 = (specimens == CONTROLS[1]) & eligible
YA, YB, transform0 = representation(X[train0][:, model_genes], X[test0][:, model_genes], N_COMPONENTS, SEED)
Y_view = np.full((len(X), YA.shape[1]), np.nan)
Y_view[train0], Y_view[test0] = YA, YB
support_rows = []
for scheme, halfwidth in widths.items():
    for s in CONTROLS:
        for k, center in enumerate(centers):
            support_rows.append(dict(scheme=scheme, specimen=s, window=k, center=center, n=int(np.sum((specimens==s) & (np.abs(z_dpt-center)<=halfwidth)))))
support = pd.DataFrame(support_rows)
support.to_csv(OUT / 'window_support.csv', index=False)
display(support.groupby(['scheme', 'specimen']).n.agg(['min', 'max']))
fig, ax = plt.subplots(figsize=(6, 4), layout='constrained')
for s, color in zip(CONTROLS, ['#0072B2', '#D55E00']):
    take = (specimens == s) & eligible
    try:
        cloud = window_clouds(Y_view[take], z_dpt[take], centers, widths['overlapping'], MIN_COUNT, MAX_POINTS, SEED)
    except ValueError as exc:
        print(s, 'historical window path unsupported:', exc)
        continue
    means = np.array([a.mean(axis=0) for a in cloud])
    ax.plot(means[:, 0], means[:, 1], 'o-', color=color, label=s)
    for a in cloud:
        ax.scatter(a[:, 0], a[:, 1], s=3, alpha=.08, color=color)
ax.set(xlabel='Training-only PC1', ylabel='Training-only PC2', title='Healthy PT distribution path on DPT scaffold')
ax.legend()
fig.savefig(OUT / 'distribution_path.pdf')
plt.show()

In [ ]:
def run_fixed_benchmark():
    frames = []
    for train_specimen in CONTROLS:
        tr = (specimens == train_specimen) & eligible
        te = (specimens != train_specimen) & eligible
        A, B, _ = representation(X[tr][:, model_genes], X[te][:, model_genes], N_COMPONENTS, SEED)
        for scheme, halfwidth in widths.items():
            try:
                frame = benchmark(A, z_dpt[tr], B, z_dpt[te], centers, halfwidth, min_count=MIN_COUNT, max_points=MAX_POINTS, seed=SEED, ridge=RIDGE, epsilon=EPSILON)
                frame['failure'] = ''
            except (ValueError, RuntimeError, np.linalg.LinAlgError) as exc:
                frame = pd.DataFrame([dict(method=method, energy_distance=np.nan, failure=str(exc)) for method in ['identity','centroid','dpt_local','flow']])
            frame['train_specimen'] = train_specimen
            frame['test_specimen'] = CONTROLS[1] if train_specimen == CONTROLS[0] else CONTROLS[0]
            frame['scheme'] = scheme
            frames.append(frame)
    return pd.concat(frames, ignore_index=True)
fixed_scores = cached_frame('fixed_dpt_flow', run_fixed_benchmark, root=OUT / 'stage_cache', params=CACHE_PARAMS, inputs={'X': X[:, model_genes], 'z': z_dpt, 'specimens': specimens, 'centers': centers, 'widths': widths}, code=CODE_DIGEST)
fixed_scores['failure'] = fixed_scores.get('failure', pd.Series('', index=fixed_scores.index)).fillna('')
fixed_scores.to_csv(OUT / 'cross_specimen_distribution_prediction.csv', index=False)
fixed_summary = fixed_scores.groupby(['scheme', 'train_specimen', 'method']).energy_distance.mean().unstack('method')
display(fixed_summary)
flow_gate = bool((fixed_scores.failure == '').all() and len(fixed_summary)==4 and (fixed_summary['flow'] < fixed_summary[['identity', 'centroid', 'dpt_local']].min(axis=1)).all())
display(fixed_scores.query("failure != ''"))
(OUT / 'experiment1_gate.json').write_text(json.dumps({'pass': flow_gate, 'rule': 'flow strictly beats all three baselines in both directions and both window schemes', 'descriptive_only': True}, indent=2))
print('Experiment 1 go/no-go:', 'OT-flow increment supported' if flow_gate else 'OT-flow increment unsupported; atlas runs independently')
fig, ax = plt.subplots(figsize=(6, 4), layout='constrained')
for method in ['identity', 'centroid', 'dpt_local', 'flow']:
    a = fixed_scores.query("scheme == 'nonoverlap' and method == @method")
    if 'z_to' not in a:
        continue
    ax.plot(a.groupby('z_to').energy_distance.mean(), marker='o', label=method)
ax.set(xlabel='Target DPT window', ylabel='Energy distance (lower is better)', title='Held-out transport: mean of two specimen directions')
ax.legend()
fig.savefig(OUT / 'cross_specimen_transport.pdf')
plt.show()

## 3. Primary experiment: frozen atlas → expression-only position

For each training direction and each of three gene folds, exclude all canonical markers and evaluation genes **before** fitting scaler/PCA. Fit a kernel-smoothed mean curve with training DPT only. Fit a second atlas with the identical mean estimator and shrinkage-stabilized full covariance. Mean projection minimizes Euclidean molecular distance; Gaussian projection uses conditional density with a uniform grid prior. Save MAP position, posterior mean, full posterior, entropy and reconstruction residual. Posteriors are conditional model uncertainty, not calibrated spatial credible intervals.

Primary projection applies the frozen atlas to **all** profiles in the other control with no coordinate or label input. To test a global specimen intercept without absorbing evaluation outcomes, split held-out profiles deterministically by structure ID into calibration and validation halves. Learn one position-independent offset from calibration **modeling features only**, penalized by a prespecified factor 4; apply it unchanged to validation profiles. Report unadjusted mapping as primary and the intercept sensitivity separately. Calibration is a transductive nuisance-estimation step, not new-specimen zero-shot prediction. It cannot fit gene evaluation residuals or position-specific effects, but offset and coordinate remain partially confounded by specimen sampling composition.

The strong three-state comparator estimates training $E[X_g|S1/S2/S3]$ and uses held-out segment labels solely in this explicit **oracle anatomy baseline**. It has access to information denied to atlas projection. Compare unseen-gene predictive errors against this baseline and supplied DPT, not random within-segment ordering. Train gene curves once on the fixed grid, then interpolate at inferred positions. Use training-only gene variance scaling. Also report within-segment errors and training-mean-normalized mean predictions; no test centering is used.

In [ ]:
def fold_design(fold, train_specimen):
    tr = (specimens == train_specimen) & np.isfinite(z_raw)
    te = specimens != train_specimen
    mg = np.flatnonzero(~marker_mask & (gene_fold != fold))
    eg = np.flatnonzero(~marker_mask & (gene_fold == fold))
    train_raw = z_raw[tr]
    train_z = (train_raw-train_raw.min()) / np.ptp(train_raw)
    A, B, transform = representation(X[tr][:, mg], X[te][:, mg], N_COMPONENTS, SEED)
    return tr, te, mg, eg, train_z, A, B, transform

def validation_split(structure_ids):
    validation = np.array([int(hashlib.sha256(s.encode()).hexdigest()[:8], 16)%2 == 1 for s in structure_ids])
    if min(validation.sum(), (~validation).sum()) < MIN_COUNT:
        raise ValueError('Insufficient offset calibration/validation profiles')
    return validation

def score_prediction(true, pred, train_values):
    var = np.maximum(train_values.var(axis=0), 1e-3)
    return float(np.mean((true-pred)**2/var))

def score_gene_curves(train_values, train_z, test_values, query_z):
    curve = conditional_gene_predictions(train_values, train_z, ATLAS_GRID, bandwidth=ATLAS_BANDWIDTH)
    prediction = interp1d(ATLAS_GRID, curve, axis=0, bounds_error=True)(np.clip(query_z, 0, 1))
    return score_prediction(test_values, prediction, train_values)

def run_inductive_atlas():
    scores, coordinates, nuisance, failures = [], [], [], []
    for fold in range(N_GENE_FOLDS):
        for train_specimen in CONTROLS:
            tr, te, mg, eg, train_z, A, B, transform = fold_design(fold, train_specimen)
            test_ids, test_labs = ids[te], labels[te]
            validation = validation_split(test_ids)
            mean_atlas = fit_mean_atlas(A, train_z, grid=ATLAS_GRID, bandwidth=ATLAS_BANDWIDTH, min_effective=MIN_COUNT)
            offset, offset_history = fit_global_offset(mean_atlas, B[~validation], penalty=OFFSET_PENALTY)
            nuisance.append(dict(fold=fold, train_specimen=train_specimen, offset_norm=float(np.linalg.norm(offset)), offset=offset.tolist(), n_calibration=int((~validation).sum()), n_validation=int(validation.sum())))
            atlas_models = {'mean_atlas': mean_atlas}
            try:
                atlas_models['gaussian_atlas'] = fit_gaussian_path(A, train_z, grid=ATLAS_GRID, bandwidth=ATLAS_BANDWIDTH, shrinkage=COV_SHRINKAGE, min_effective=MIN_COUNT)
            except (ValueError, RuntimeError, np.linalg.LinAlgError) as exc:
                failures.append(dict(fold=fold, train_specimen=train_specimen, method='gaussian_atlas', reason=str(exc)))
            gene_curve = conditional_gene_predictions(X[tr][:, eg], train_z, ATLAS_GRID, bandwidth=ATLAS_BANDWIDTH)
            gene_interp = interp1d(ATLAS_GRID, gene_curve, axis=0, bounds_error=True)
            predictions_by_model = {}
            for method, atlas in atlas_models.items():
                projector = project_mean_atlas if method=='mean_atlas' else project_gaussian_path
                for adjustment, feature_values in [('none', B), ('calibrated_global_intercept', B-offset)]:
                    projected = projector(atlas, feature_values)
                    zz, probabilities = projected['z_MAP'], projected['posterior']
                    prediction = gene_interp(zz)
                    predictions_by_model[(method, adjustment)] = prediction
                    for scope, use in [('all_test', np.ones(len(B), bool)), ('validation', validation)]:
                        if adjustment != 'none' and scope=='all_test':
                            continue
                        scores.append(dict(fold=fold, train_specimen=train_specimen, method=method, adjustment=adjustment, scope=scope, segment='all', n=int(use.sum()), gene_mse=score_prediction(X[te][:, eg][use], prediction[use], X[tr][:, eg])))
                        for lab in ['S1','S2','S3']:
                            sub = use & (test_labs==lab)
                            if sub.any():
                                scores.append(dict(fold=fold, train_specimen=train_specimen, method=method, adjustment=adjustment, scope=scope, segment=lab, n=int(sub.sum()), gene_mse=score_prediction(X[te][:, eg][sub], prediction[sub], X[tr][:, eg])))
                    for j, sid in enumerate(test_ids):
                        coordinates.append(dict(fold=fold, train_specimen=train_specimen, test_specimen=specimens[te][j], method=method, adjustment=adjustment, structure_id=sid, validation=bool(validation[j]), z=float(zz[j]), z_mean=float(projected['z_mean'][j]), entropy=float(projected['entropy'][j]), residual=float(projected['residual_norm'][j]), **{f'p_{k:03d}':float(p) for k,p in enumerate(probabilities[j])}))
            segment_prediction = segment_means(X[tr][:, eg], labels[tr], test_labs)['prediction']
            test_dpt = (z_raw[te]-z_raw[tr].min()) / np.ptp(z_raw[tr])
            dpt_available = np.isfinite(test_dpt)
            dpt_prediction = gene_interp(np.clip(test_dpt[dpt_available],0,1))
            for scope, use in [('all_test', np.ones(len(B), bool)), ('validation', validation)]:
                scores.append(dict(fold=fold, train_specimen=train_specimen, method='segment_only_oracle', adjustment='none', scope=scope, segment='all', n=int(use.sum()), gene_mse=score_prediction(X[te][:, eg][use], segment_prediction[use], X[tr][:, eg])))
                for lab in ['S1','S2','S3']:
                    sub = use & (test_labs == lab)
                    if sub.any():
                        scores.append(dict(fold=fold, train_specimen=train_specimen, method='segment_only_oracle', adjustment='none', scope=scope, segment=lab, n=int(sub.sum()), gene_mse=score_prediction(X[te][:, eg][sub], segment_prediction[sub], X[tr][:, eg])))
                common = use & dpt_available
                scores.append(dict(fold=fold, train_specimen=train_specimen, method='DPT_supplied', adjustment='none', scope=scope+'_DPT_common', segment='all', n=int(common.sum()), gene_mse=score_prediction(X[te][:, eg][common], dpt_prediction[use[dpt_available]], X[tr][:, eg])))
                for method in ['mean_atlas', 'gaussian_atlas']:
                    for adjustment in ['none','calibrated_global_intercept']:
                        if adjustment!='none' and scope=='all_test':
                            continue
                        pred = predictions_by_model.get((method, adjustment))
                        if pred is None:
                            continue
                        scores.append(dict(fold=fold, train_specimen=train_specimen, method=method, adjustment=adjustment, scope=scope+'_DPT_common', segment='all', n=int(common.sum()), gene_mse=score_prediction(X[te][:, eg][common], pred[common], X[tr][:, eg])))
            print(f'Atlas fold {fold}, train {train_specimen}: projected {len(B)} profiles without test DPT/labels', flush=True)
    return {'scores':pd.DataFrame(scores).to_json(orient='table'), 'coordinates':pd.DataFrame(coordinates).to_json(orient='table'), 'offsets':pd.DataFrame(nuisance).to_json(orient='table'), 'failures':pd.DataFrame(failures).to_json(orient='table')}

import io
atlas_payload = cached_payload('inductive_atlas', run_inductive_atlas, root=OUT/'stage_cache', params=CACHE_PARAMS, inputs={'X':X,'z_raw':z_raw,'ids':ids,'labels_evaluation_only':labels,'specimens':specimens,'gene_fold':gene_fold,'markers':marker_mask}, code=CODE_DIGEST)
atlas_scores = pd.read_json(io.StringIO(str(atlas_payload['scores'])), orient='table')
atlas_coordinates = pd.read_json(io.StringIO(str(atlas_payload['coordinates'])), orient='table')
atlas_offsets = pd.read_json(io.StringIO(str(atlas_payload['offsets'])), orient='table')
atlas_failures = pd.read_json(io.StringIO(str(atlas_payload['failures'])), orient='table')
for name, frame in [('inductive_gene_scores',atlas_scores),('inductive_positions_and_posteriors',atlas_coordinates),('calibration_offsets',atlas_offsets),('inductive_failures',atlas_failures)]:
    frame.to_csv(OUT/(name+'.csv'), index=False)
display(atlas_scores.query("scope == 'validation' and segment == 'all'").groupby(['method','adjustment']).gene_mse.agg(['mean','min','max']))
display(atlas_offsets.drop(columns='offset'))
display(atlas_failures)

## 4. Distribution evolution beyond movement of the mean

Fit a Gaussian path with all training profiles and fixed kernel bandwidth. Covariance uses a prespecified 25% diagonal shrinkage and an eigenvalue floor. Effective kernel sample size is checked at every grid point; sparse support is not fabricated. Use a smooth cubic mean and piecewise-linear SPD covariance interpolation; solve the symmetric Lyapunov equation within each interval and integrate the Gaussian velocity.

Compare identity, historical raw centroid increments, **smooth mean-only translation** and covariance-aware Gaussian transport on identical held-out clouds, in both window schemes and specimen directions. Test DPT defines source/target windows for this diagnostic only, never for atlas projection. Record energy distance, mean error, covariance error and centered-cloud energy to distinguish mean improvement from shape improvement. Hold the mean estimator fixed when comparing mean-only and Gaussian transport. No failed field blocks mean-atlas inference.

In [ ]:
def run_gaussian_transport():
    frames = []
    for train_specimen in CONTROLS:
        tr = (specimens==train_specimen) & eligible
        te = (specimens!=train_specimen) & eligible
        A,B,_ = representation(X[tr][:,model_genes],X[te][:,model_genes],N_COMPONENTS,SEED)
        for scheme,halfwidth in widths.items():
            try:
                frame = gaussian_transport_benchmark(A,z_dpt[tr],B,z_dpt[te],centers,halfwidth,bandwidth=ATLAS_BANDWIDTH*(np.nanmax(z_dpt[tr])-np.nanmin(z_dpt[tr])),shrinkage=COV_SHRINKAGE,min_count=MIN_COUNT,max_points=MAX_POINTS,seed=SEED)
                frame['failure'] = ''
            except (ValueError, RuntimeError, np.linalg.LinAlgError) as exc:
                frame = pd.DataFrame([dict(method=method, energy_distance=np.nan, failure=str(exc)) for method in ['smooth_mean','gaussian_flow']])
            frame['scheme'],frame['train_specimen'] = scheme,train_specimen
            frames.append(frame)
    return pd.concat(frames,ignore_index=True)
gaussian_scores = cached_frame('gaussian_transport',run_gaussian_transport,root=OUT/'stage_cache',params=CACHE_PARAMS,inputs={'X':X[:,model_genes],'z':z_dpt,'specimens':specimens,'eligible':eligible,'centers':centers,'widths':widths},code=CODE_DIGEST)
gaussian_scores['failure'] = gaussian_scores.get('failure', pd.Series('', index=gaussian_scores.index)).fillna('')
gaussian_scores.to_csv(OUT/'gaussian_mean_vs_covariance_transport.csv',index=False)
gaussian_summary = gaussian_scores.groupby(['scheme','train_specimen','method']).energy_distance.mean().unstack('method')
display(gaussian_summary)
gaussian_flow_gate = bool((gaussian_scores.failure == '').all() and len(gaussian_summary)==4 and (gaussian_summary['gaussian_flow'] < gaussian_summary['smooth_mean']).all())
display(gaussian_scores.query("failure != ''"))
print('Covariance-flow increment supported across both specimens and window schemes:',gaussian_flow_gate)
fig,ax = plt.subplots(figsize=(6,4),layout='constrained')
for method in ['centroid','smooth_mean','gaussian_flow']:
    group = gaussian_scores.query("scheme == 'nonoverlap' and method == @method")
    if 'z_to' not in group:
        continue
    ax.plot(group.groupby('z_to').energy_distance.mean(),marker='o',label=method)
ax.set(xlabel='Target DPT window (diagnostic only)',ylabel='Held-out energy distance',title='Does covariance transport add to the same smooth mean?')
ax.legend()
fig.savefig(OUT/'gaussian_transport_ablation.pdf')
plt.show()

## 5. Projection validation: withheld anatomy, physical depth, markers and gene curves

Only now read physical anchors and held-out segment labels. Compute nearest and mean-three-glomeruli distances per specimen after validating reviewed glomerular centroids against the current segmentation. These are cortical-depth proxies, not longitudinal PT ground truth.

For each gene fold, method, and adjustment, evaluate rank association with held-out segment order, segment median positions, all pairwise S1/S2/S3 ordering concordance, and physical-depth correlation. Report all eligible targets and the matched DPT subset separately. The global-intercept sensitivity uses validation profiles only. No tubule-level p-values or pooling the two specimens as independent structures. These checks do not select model settings.

Plot excluded PT markers and held-out nonmarker genes against **new frozen-atlas positions**, alongside supplied DPT. Curves inferred in the held-out specimen are descriptive visualization; prediction errors above use only training gene curves. Save train-predicted and held-out-smoothed curves separately, including per-gene cross-specimen curve agreement. Sex-dimorphic S3 markers may legitimately differ. Upstream DPT remains marker-informed, so these diagnostics are not proof of independent spatial ground truth.

In [ ]:
def physical_depth():
    if not GLOMERULUS_PATH.is_file():
        raise FileNotFoundError(f'Reviewed glomeruli required: {GLOMERULUS_PATH}')
    anchors = ad.read_h5ad(GLOMERULUS_PATH)
    needed={'sample','coarse_class','feature_index','x_centroid','y_centroid'}
    if not needed.issubset(anchors.obs):
        raise ValueError(f'Glomerulus export missing {needed-set(anchors.obs)}')
    result=np.full((len(X),2),np.nan)
    for s in CONTROLS:
        a=anchors.obs.loc[(anchors.obs['sample'].astype(str)==s)&(anchors.obs.coarse_class.astype(str)=='Glomerulus')]
        if a.empty:
            raise ValueError(f'{s}: no reviewed glomeruli')
        _read_and_validate_segmentation(DATA_ROOT,s,a)
        use=specimens==s
        distance,_=cKDTree(a[['x_centroid','y_centroid']].to_numpy(float)).query(xy[use],k=min(3,len(a)))
        distance=np.asarray(distance).reshape(use.sum(),-1)
        result[use,0],result[use,1]=distance[:,0],distance.mean(axis=1)
    return result

def ordering_concordance(z,lab):
    values=[]
    for early,late in [('S1','S2'),('S1','S3'),('S2','S3')]:
        a,b=z[lab==early],z[lab==late]
        if len(a) and len(b):
            delta=b[:,None]-a[None,:]
            values.append(float(np.mean((delta>0)+.5*(delta==0))))
    return float(np.mean(values)) if values else np.nan

depth=physical_depth()
validation_rows,segment_rows=[],[]
for key,group in atlas_coordinates.groupby(['fold','train_specimen','method','adjustment']):
    ix=pd.Index(ids).get_indexer(group.structure_id)
    allowed=np.ones(len(group),bool) if key[3]=='none' else group.validation.to_numpy(bool)
    for scope,use in [('all_projection',allowed),('DPT_common',allowed&eligible[ix])]:
        zz=group.z.to_numpy()[use]
        labs=labels[ix][use]
        rank=np.array([{'S1':0,'S2':1,'S3':2}[a] for a in labs])
        base=dict(fold=key[0],train_specimen=key[1],test_specimen=group.test_specimen.iloc[0],method=key[2],adjustment=key[3],scope=scope,n=int(use.sum()))
        validation_rows.append(dict(**base,segment_spearman=spearmanr(zz,rank).statistic,ordering_concordance=ordering_concordance(zz,labs),nearest_glomerulus=spearmanr(zz,depth[ix[use],0]).statistic,mean_three_glomeruli=spearmanr(zz,depth[ix[use],1]).statistic))
        for lab in ['S1','S2','S3']:
            take=labs==lab
            if take.any():
                segment_rows.append(dict(**base,segment=lab,median_z=float(np.median(zz[take])),n_segment=int(take.sum())))
for s in CONTROLS:
    use=(specimens==s)&eligible
    rank=np.array([{'S1':0,'S2':1,'S3':2}[a] for a in labels[use]])
    validation_rows.append(dict(fold=-1,train_specimen='supplied',test_specimen=s,method='DPT_supplied',adjustment='none',scope='DPT_common',n=int(use.sum()),segment_spearman=spearmanr(z_dpt[use],rank).statistic,ordering_concordance=ordering_concordance(z_dpt[use],labels[use]),nearest_glomerulus=spearmanr(z_dpt[use],depth[use,0]).statistic,mean_three_glomeruli=spearmanr(z_dpt[use],depth[use,1]).statistic))
projection_validation=pd.DataFrame(validation_rows)
projection_validation.to_csv(OUT/'projection_anatomy_and_depth.csv',index=False)
pd.DataFrame(segment_rows).to_csv(OUT/'withheld_segment_positions.csv',index=False)
display(projection_validation.query("scope == 'DPT_common'").groupby(['test_specimen','method','adjustment'])[['segment_spearman','ordering_concordance','nearest_glomerulus','mean_three_glomeruli']].mean())

In [ ]:
curve_rows,agreement_rows=[],[]
for train_specimen in CONTROLS:
    tr,te,mg,eg,train_z,A,B,transform=fold_design(0,train_specimen)
    panel=np.r_[np.flatnonzero(marker_mask),eg]
    train_curve=conditional_gene_predictions(X[tr][:,panel],train_z,ATLAS_GRID,bandwidth=ATLAS_BANDWIDTH)
    for method in ['mean_atlas','gaussian_atlas']:
        group=atlas_coordinates.query("fold == 0 and train_specimen == @train_specimen and method == @method and adjustment == 'none'").set_index('structure_id')
        if len(group)!=int(te.sum()):
            continue
        zz=group.loc[ids[te],'z'].to_numpy()
        test_curve=conditional_gene_predictions(X[te][:,panel],zz,ATLAS_GRID,bandwidth=ATLAS_BANDWIDTH)
        for j,g in enumerate(panel):
            is_marker=bool(marker_mask[g])
            a,b=train_curve[:,j],test_curve[:,j]
            correlation=np.corrcoef(a,b)[0,1] if np.ptp(a)>0 and np.ptp(b)>0 else np.nan
            agreement_rows.append(dict(train_specimen=train_specimen,method=method,gene=genes[g],marker=is_marker,curve_correlation=correlation))
            if is_marker or j < len(np.flatnonzero(marker_mask))+6:
                for center,aa,bb in zip(ATLAS_GRID,a,b):
                    curve_rows.append(dict(train_specimen=train_specimen,test_specimen=CONTROLS[1] if train_specimen==CONTROLS[0] else CONTROLS[0],method=method,gene=genes[g],marker=is_marker,z=center,training_prediction=aa,heldout_smoothed=bb))
    supplied = (z_raw[te]-z_raw[tr].min())/np.ptp(z_raw[tr])
    has_dpt = np.isfinite(supplied)
    dpt_curve = conditional_gene_predictions(X[te][:,panel][has_dpt],np.clip(supplied[has_dpt],0,1),ATLAS_GRID,bandwidth=ATLAS_BANDWIDTH)
    for j,g in enumerate(panel):
        is_marker=bool(marker_mask[g])
        a,b=train_curve[:,j],dpt_curve[:,j]
        correlation=np.corrcoef(a,b)[0,1] if np.ptp(a)>0 and np.ptp(b)>0 else np.nan
        agreement_rows.append(dict(train_specimen=train_specimen,method='DPT_supplied',gene=genes[g],marker=is_marker,curve_correlation=correlation))
        if is_marker or j < len(np.flatnonzero(marker_mask))+6:
            for center,aa,bb in zip(ATLAS_GRID,a,b):
                curve_rows.append(dict(train_specimen=train_specimen,test_specimen=CONTROLS[1] if train_specimen==CONTROLS[0] else CONTROLS[0],method='DPT_supplied',gene=genes[g],marker=is_marker,z=center,training_prediction=aa,heldout_smoothed=bb))
curve_table=pd.DataFrame(curve_rows)
curve_table.to_csv(OUT/'new_coordinate_marker_and_gene_curves.csv',index=False)
curve_agreement=pd.DataFrame(agreement_rows)
curve_agreement.to_csv(OUT/'heldout_gene_curve_agreement.csv',index=False)
display(curve_agreement.groupby(['method','marker']).curve_correlation.agg(['median','mean','count']))
for gene in ['Slc5a2','Slc22a6','Slc7a13']:
    fig,ax=plt.subplots(figsize=(6,3.5),layout='constrained')
    for (s,method),group in curve_table[curve_table.gene.str.casefold()==gene.casefold()].groupby(['test_specimen','method']):
        ax.plot(group.z,group.heldout_smoothed,linestyle='--' if method=='DPT_supplied' else '-',label=s+' '+method)
    ax.set(xlabel='Frozen-atlas position',ylabel='Log-normalized expression',title=gene+' — withheld from reconstruction')
    ax.legend(fontsize=7)
    fig.savefig(OUT/(gene+'_new_atlas_coordinate.pdf'))
    plt.show()

## 6. Mean-atlas bootstrap stability (independent of flow)

Bootstrap training tubules, reconstruction genes, and both. Refit scaler/PCA and the mean atlas each time, then project the unchanged other control without its DPT, labels or offset calibration. Keep sparse-support failures explicit. Report position standard deviation and rank agreement with the full-training atlas; this measures sampling/feature sensitivity, distinct from the Gaussian posterior entropy. No initialization search is needed for this fixed-training-coordinate mean atlas. Fully latent training-coordinate refinement remains a later hypothesis rather than mixing it into the first inductive test.

Retain per-profile bootstrap count, mean, SD and range. Most profiles can be stable while a minority jump far along the axis; report that tail explicitly. The SD > 0.1 summary is an exploratory sensitivity flag, not a prevalidated acceptance threshold.

In [ ]:
def run_mean_stability():
    rng=np.random.default_rng(SEED)
    rows=[]
    for train_specimen in CONTROLS:
        tr,te,mg,eg,train_z,A,B,transform=fold_design(0,train_specimen)
        train_rows=np.flatnonzero(tr)
        for scheme in ['tubules','genes','both']:
            for repeat in range(N_BOOT):
                rr=rng.choice(len(train_rows),len(train_rows),replace=True) if scheme in ['tubules','both'] else np.arange(len(train_rows))
                gg=rng.choice(mg,len(mg),replace=True) if scheme in ['genes','both'] else mg
                try:
                    AA,BB,_=representation(X[train_rows[rr]][:,gg],X[te][:,gg],N_COMPONENTS,SEED)
                    atlas=fit_mean_atlas(AA,train_z[rr],grid=ATLAS_GRID,bandwidth=ATLAS_BANDWIDTH,min_effective=MIN_COUNT)
                    projected=project_mean_atlas(atlas,BB)
                    for sid,zz in zip(ids[te],projected['z_MAP']):
                        rows.append(dict(train_specimen=train_specimen,scheme=scheme,repeat=repeat,structure_id=sid,z=float(zz),failure=''))
                except (ValueError,RuntimeError,np.linalg.LinAlgError) as exc:
                    rows.append(dict(train_specimen=train_specimen,scheme=scheme,repeat=repeat,structure_id='',z=np.nan,failure=str(exc)))
            print('Bootstrap completed:',train_specimen,scheme,flush=True)
    return pd.DataFrame(rows)
stability=cached_frame('mean_atlas_stability',run_mean_stability,root=OUT/'stage_cache',params={**CACHE_PARAMS,'n_boot':N_BOOT},inputs={'X':X,'z_raw':z_raw,'ids':ids,'specimens':specimens,'model_genes':model_genes},code=CODE_DIGEST)
stability['failure']=stability.failure.fillna('')
stability.to_csv(OUT/'mean_atlas_bootstrap_positions.csv',index=False)
position_uncertainty=stability.query("failure == ''").groupby(['train_specimen','scheme','structure_id']).z.agg(['count','mean','std','min','max']).reset_index()
position_uncertainty.to_csv(OUT/'mean_atlas_per_profile_bootstrap_uncertainty.csv',index=False)
stability_summary=position_uncertainty.groupby(['train_specimen','scheme'])['std'].agg(median='median',max='max',fraction_sd_above_point_one=lambda v:float(np.mean(v>.1)))
stability_summary.to_csv(OUT/'mean_atlas_stability_summary.csv')
display(stability_summary)
print('SD > 0.1 is a descriptive sensitivity flag, not a calibrated confidence cutoff')
display(stability.query("failure != ''"))

## 7. Separate scientific decisions; downstream interpretation deferred

The atlas hypothesis is assessed using held-out gene prediction, the strong segment-only comparator, held-out segment order, physical depth, excluded markers, curve agreement and stability. The Gaussian-flow hypothesis is assessed by incremental transport improvement over **its identical mean path**, including centered shape metrics. DPT agreement is not success. Two controls and depth proxies cannot establish true longitudinal location or calibrated uncertainty.

Record prespecified descriptive gene-prediction screens, but do not automatically declare a validated coordinate from a single metric. A global-offset sensitivity is not the primary zero-shot method and may confound sampling composition with position. A Gaussian likelihood can improve expression fit while moving profiles incorrectly; examine depth and ordering alongside it.

AKI and TF Jacobian interpretation are deferred in this version. The previous weak atlas moment-blending and OT Jacobian branches are superseded; no field interpretation follows simply from a smooth plot. The next step depends on evidence: pursue a mean atlas if it maps held-out biology; add probabilistic shape/flow only if it earns an incremental predictive benefit.

In [ ]:
primary_scores=atlas_scores.query("scope == 'validation' and segment == 'all' and adjustment == 'none'")
primary_pivot=primary_scores.pivot(index=['fold','train_specimen'],columns='method',values='gene_mse')
mean_vs_segment=bool((primary_pivot['mean_atlas']<primary_pivot['segment_only_oracle']).all())
gaussian_vs_mean=bool('gaussian_atlas' in primary_pivot and (primary_pivot['gaussian_atlas']<primary_pivot['mean_atlas']).all())
decision={'mean_gene_screen_beats_segment_oracle_all_folds_and_directions':mean_vs_segment,'gaussian_mapping_gene_screen_beats_mean_all_folds_and_directions':gaussian_vs_mean,'gaussian_flow_beats_same_smooth_mean_all_schemes_and_directions':gaussian_flow_gate,'historical_ot_flow_passed':flow_gate,'biological_validation_requires_review':['physical_depth','segment_order','withheld_genes_and_markers','sampling_and_feature_stability'],'AKI_and_TF':'deferred'}
(OUT/'atlas_first_decision.json').write_text(json.dumps(decision,indent=2))
display(primary_pivot)
print(json.dumps(decision,indent=2))